# Integration of lakes into the Hydrography90m stream network

## A workflow using `hydrographr` lake functions

# Introduction

The following workflow showcases the integration of lakes into the stream network and perform 
basic network analyses of the lakes connected to the stream network. This includes:

     1. Identifying lakes in our study area
     2. Identifying the intersection points between stream segments and lakes
     3. Delineating the upstream contributing catchment area from a lakes outlet point
     4. Calculating the land cover change of a lakes catchment area
     5. Calculating the upstream and downstream distance from a lakes outlet
     6. Calculating the distance between two lakes
 
Specifically made for this usecase, we will apply the three hydrographr (Schürz, M. & Grigoropoulou, A., 2023) 
lake functions:

- (i) `extract_lake_ids()`
- (ii) `get_lake_intersection()`
- (iii) `get_lake_catchment()`  
     
We will use freshwater fish species occurrence data from the  and 
the HydroLAKES dataset (Messager et al., 2016) and integrate them into the 
Hydrography90m stream network covering parts of the country of Greece (Amatulli et al., 2022).

Load required libraries

In [1]:
library(hydrographr)
library(data.table)
library(dplyr)
library(igraph)
library(terra)
library(tools)
library(stringr)
library(leaflet)
library(leafem)
library(sf)
library(tidyr)
library(readr)
library(ggplot2)

Warning message:
“replacing previous import ‘data.table::first’ by ‘dplyr::first’ when loading ‘hydrographr’”
Warning message:
“replacing previous import ‘data.table::between’ by ‘dplyr::between’ when loading ‘hydrographr’”
Warning message:
“replacing previous import ‘data.table::last’ by ‘dplyr::last’ when loading ‘hydrographr’”
Warning message:
“replacing previous import ‘dplyr::as_data_frame’ by ‘igraph::as_data_frame’ when loading ‘hydrographr’”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘igraph’


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:



# Input data for lake integration

### Species data

For the spatial reference and to integrate lakes into our stream network we will continue to use our previously snapped freshwater fish dataset.

In [3]:
gbif_snap_result <-  fread("./points_snapped/gbif_snapped_min_strahler2_dist400.csv")

In [4]:
str(gbif_snap_result)
head(gbif_snap_result)

Classes ‘data.table’ and 'data.frame':	1004 obs. of  8 variables:
 $ gbifID                   :integer64 1609378558 1609379380 883493441 4453890552 1609378811 3333112020 5828528625 5021107221 ... 
 $ subc_id                  : int  562024980 561296307 562518639 514712114 561847341 560241456 561240469 562384312 562468470 560839311 ...
 $ strahler                 : int  5 4 4 5 5 4 5 5 5 4 ...
 $ decimalLongitude_snapped : num  22.1 21.6 25 19.8 21.7 ...
 $ decimalLongitude_original: num  22.1 21.6 25 19.8 21.7 ...
 $ decimalLatitude_snapped  : num  37 38.4 34.9 39.7 37.4 ...
 $ decimalLatitude_original : num  37 38.4 34.9 39.7 37.4 ...
 $ distance_metres          : num  185 326 304 117 120 ...
 - attr(*, ".internal.selfref")=<externalptr> 


gbifID,subc_id,strahler,decimalLongitude_snapped,decimalLongitude_original,decimalLatitude_snapped,decimalLatitude_original,distance_metres
<int64>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1609378558,562024980,5,22.07792,22.0800,37.03000,37.0300,185.3681
1609379380,561296307,4,21.62958,21.6300,38.35292,38.3500,325.8013
883493441,562518639,4,24.98292,24.9808,34.93542,34.9333,304.1995
4453890552,514712114,5,19.84707,19.8479,39.67373,39.6729,116.9308
1609378811,561847341,5,21.68125,21.6800,37.38042,37.3800,119.9782
3333112020,560241456,4,23.36042,23.3592,40.03708,40.0371,103.8561


### Hydrography90m dataset

In order for our lake functions to work we will need some additional files of the **Hydrography90m dataset**, in addition to the already downloaded Hydrography90m stream network:

for `get_lake_intersection()`:
1. basin ("basin") raster files
2. stream network ("segment") raster files
3. flow accumulation (`"accumulation"`) raster files

for `get_lake_catchment()`:
4. flow direction (`"direction"`) raster files

for 'land cover analysis':
5. sub-catchment (`"sub_catchment"`) files

First, we identify the tile ID covering our species data. In our case that are two tiles (h18v04; h20v04).
**NOTE:** check also visually which tiles overlap with your study area to avoid unecessary tile downloads.

In [8]:
# get tile ID for Greece
# tile_id <- get_tile_id(data = gbif_snap_result,
#                       lon = "decimalLongitude_snapped",
#                       lat = "decimalLatitude_snapped")

tile_id <- c("h18v04", "h20v04")

In [16]:
# Note: This will take some time. Instead, we will load the prepared data from disk

# select the necessary hydrological variables for downloading
# vars_tif <- c("basin", "sub_catchment", "segment", "accumulation", "direction")

# Extend timeout to 1000s to allow uninterrupted downloading
# options(timeout = 1000)

# Download the tif.files of the selected variables
# download_tiles(variable = vars_tif,
#               tile_id = tile_id,
#               file_format = "tif",
#               download_dir = "./lakes")

Note that the files have been downloaded to a new folder (i.e., “r.watershed”).

Before we merge, we are going to crop the raster tiles to the extent of the stream network of the study area.

Lets create a folder to store our lake outputs

In [2]:
if (!dir.exists(file.path("./lakes"))) dir.create(file.path("./lakes"))

read_dir <- "/home/participant/data_readonly/"

In [10]:
# load in stream network
stream_network <- st_read("./spatial/stream_networks/partial_stream_network.gpkg")

Reading layer `partial_stream_network' from data source 
  `/home/participant/data_write/spatial/stream_networks/partial_stream_network.gpkg' 
  using driver `GPKG'
Simple feature collection with 125209 features and 5 fields (with 16 geometries empty)
Geometry type: LINESTRING
Dimension:     XY
Bounding box:  xmin: 19.3121 ymin: 34.9354 xmax: 28.2346 ymax: 42.8104
Geodetic CRS:  WGS 84


In [12]:
# Get the full paths of the raster tiles
raster_dir <- list.files(paste0(read_dir, "lakes/r.watershed"), pattern = ".tif",
                         full.names = TRUE, recursive = TRUE)

raster_dir

[1] "/home/participant/data_readonly/lakes/r.watershed/accumulation_tiles20d/accumulation_h18v04.tif"  
 [2] "/home/participant/data_readonly/lakes/r.watershed/accumulation_tiles20d/accumulation_h20v04.tif"  
 [3] "/home/participant/data_readonly/lakes/r.watershed/basin_tiles20d/basin_h18v04.tif"                
 [4] "/home/participant/data_readonly/lakes/r.watershed/basin_tiles20d/basin_h20v04.tif"                
 [5] "/home/participant/data_readonly/lakes/r.watershed/direction_tiles20d/direction_h18v04.tif"        
 [6] "/home/participant/data_readonly/lakes/r.watershed/direction_tiles20d/direction_h20v04.tif"        
 [7] "/home/participant/data_readonly/lakes/r.watershed/segment_tiles20d/segment_h18v04.tif"            
 [8] "/home/participant/data_readonly/lakes/r.watershed/segment_tiles20d/segment_h20v04.tif"            
 [9] "/home/participant/data_readonly/lakes/r.watershed/sub_catchment_tiles20d/sub_catchment_h18v04.tif"
[10] "/home/participant/data_readonly/lakes/r.watershed/sub_catchment_tiles20d/sub_catchment_h20v04.tif"

In [13]:
# Get the extent from the stream network
bbox <- ext(stream_network)

for (itile in raster_dir) {
  r <- rast(itile)
  original_dtype <- datatype(r)
  r_cropped <- crop(r, bbox)
  writeRaster(r_cropped,
              paste0("./lakes", "/", str_remove(basename(itile), ".tif"), "_tmp.tif"),
              datatype = original_dtype,
              NAflag = 0,
              overwrite = TRUE)
}

In [14]:
# Create a new data folder in the working directory to store all the data
dir.create(paste0("./lakes/", "hydrography90m"))

Now we can merge all tif raster files covering our study area.

In [17]:
# select the necessary hydrological variables
vars_tif <- c("basin", "sub_catchment", "segment", "accumulation", "direction")

# merge all raster files of the respective variables
for (var in vars_tif) {
  # Define the folder for the current variable
  var_dir <- "./lakes"

  # Build path to .tif file names for each variable and tile ids
  tile_files <- paste0(var, "_", tile_id, "_tmp.tif")

  # List only the matching .tif files in that folder
  tile_names <- list.files(var_dir, pattern = "\\_tmp.tif$", full.names = FALSE)
  tile_names <- tile_names[tile_names %in% tile_files]

  # Run the merge function
  merge_tiles(
    tile_dir = var_dir,
    tile_names = tile_names,
    out_dir = paste0("./lakes", "/hydrography90m"),
    file_name = paste0("merge_", var, ".tif"),
    read = FALSE,
    bigtiff = TRUE
  )
}

Merged file saved under:  ./lakes/hydrography90m 
Merged file saved under:  ./lakes/hydrography90m 
Merged file saved under:  ./lakes/hydrography90m 
Merged file saved under:  ./lakes/hydrography90m 
Merged file saved under:  ./lakes/hydrography90m 


It is always good practice to clean disk space by removing the files and folders we don’t need anymore.

In [44]:
# remove folder
# unlink("r.watershed", recursive = TRUE)
# add here to remove temp files

## HydroLAKES dataset

Our lake functions allow us to integrate any spatial vector polygon lake dataset. For today we will use a already downloaded subset of the global **HydroLAKES dataset**.
For more information on the dataset you can have a look [here](https://www.hydrosheds.org/products/hydrolakes)

## Identifying lakes within our study area

First we will identify all lakes of the HydroLAKES dataset located within our study area. We will do this by using the hydrographr 
`extract_lake_ids()` function. When setting the bounding box parameter `bbox = TRUE` the function will use the maximum and minimum spatial 
extent of the provided species occurrence points as input for the bounding box and return all lake IDs located inside it in table format (lake_id.txt).

In [18]:
# first define the path to the lake dataset
hydro_lakes <- paste0(read_dir, "lakes/spatial/hydrolakes_bbox.gpkg")

extract_lake_ids(data = gbif_snap_result,
                 lon = "decimalLongitude_snapped",
                 lat = "decimalLatitude_snapped",
                 bbox = TRUE,
                 lake_shape = hydro_lakes,
                 lake_id_table = "./lakes",
                 quiet = FALSE)

# load in results
lake_id <- fread(paste0("./lakes", "/lake_id.txt"), header = TRUE)
head(lake_id)

Hylak_id
<int>
172993
1367380
1367379
1368249
173286
173207
173248
1367963
1367861


Hylak_id
<int>
172993
1367380
1367379
1368249
173286
173207


This results in 434 lakes located inside our study area. Lets rename the resulting lake_id.txt file in order to save it for later and rerun the function
using other parameter settings.

In [19]:
# rename the lake_id.txt file to lake_id_bbox.txt
file.rename(paste0("./lakes", "/lake_id.txt"), paste0("./lakes", "/lake_id_bbox.txt"))

[1] TRUE

![](./img/lakes_species_bbox.png)

With the `extract_lake_ids()` function it is also possible to retrieve only lakes containing species occurrence points. We simply set the
bounding box parameter `bbox = FALSE` and the function returns all lake IDs that have occurrence points located within them.

In [20]:
Sys.setenv(PROJ_DATA = "/opt/conda/share/proj")

In [21]:
# extract lakes which contain species occurrence points
extract_lake_ids(data = gbif_snap_result,
                 lon = "decimalLongitude_snapped",
                 lat = "decimalLatitude_snapped",
                 bbox = FALSE,
                 lake_shape = hydro_lakes,
                 lake_id_table = "./lakes",
                 quiet = FALSE)

Warning 6: Normalized/laundered field name: 'diss' to 'diss_1'
INFO: Open of `/tmp/Rtmpxkx6L8/lakes.shp'
      using driver `ESRI Shapefile' successful.
INFO: Open of `/tmp/Rtmpxkx6L8/lakes.shp'
      using driver `ESRI Shapefile' successful.
0...10...20...30...40...50...60...70...80...90...100 - done.


decimalLongitude_snapped,decimalLatitude_snapped,Hylak_id
<dbl>,<dbl>,<int>
22.07792,37.03000,0
21.62958,38.35292,0
24.98292,34.93542,0
19.84707,39.67373,0
21.68125,37.38042,0
23.36042,40.03708,0
22.42958,38.43070,0
24.09625,35.58958,0
24.47458,35.15292,0


In [30]:
# lets check how many species are found in lakes
# loading in results
lake_id <- fread("./lakes/lake_id.txt", header = TRUE)

# add ID column in case it is missing for merging tables
lake_id$id <- seq_len(nrow(lake_id))

head(lake_id)
str(lake_id)

decimalLongitude_snapped,decimalLatitude_snapped,Hylak_id,id
<dbl>,<dbl>,<int>,<int>
22.07792,37.03000,0,1
21.62958,38.35292,0,2
24.98292,34.93542,0,3
19.84707,39.67373,0,4
21.68125,37.38042,0,5
23.36042,40.03708,0,6


Classes ‘data.table’ and 'data.frame':	1004 obs. of  4 variables:
 $ decimalLongitude_snapped: num  22.1 21.6 25 19.8 21.7 ...
 $ decimalLatitude_snapped : num  37 38.4 34.9 39.7 37.4 ...
 $ Hylak_id                : int  0 0 0 0 0 0 0 0 0 0 ...
 $ id                      : int  1 2 3 4 5 6 7 8 9 10 ...
 - attr(*, ".internal.selfref")=<externalptr> 


In [26]:
# add ID column in case it is missing for merging tables
gbif_snap_result$id <- seq_len(nrow(gbif_snap_result))
head(gbif_snap_result)

gbifID,subc_id,strahler,decimalLongitude_snapped,decimalLongitude_original,decimalLatitude_snapped,decimalLatitude_original,distance_metres,id
<int64>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1609378558,562024980,5,22.07792,22.0800,37.03000,37.0300,185.3681,1
1609379380,561296307,4,21.62958,21.6300,38.35292,38.3500,325.8013,2
883493441,562518639,4,24.98292,24.9808,34.93542,34.9333,304.1995,3
4453890552,514712114,5,19.84707,19.8479,39.67373,39.6729,116.9308,4
1609378811,561847341,5,21.68125,21.6800,37.38042,37.3800,119.9782,5
3333112020,560241456,4,23.36042,23.3592,40.03708,40.0371,103.8561,6


In [29]:
lake_id

Hylak_id
<int>
172993
1367380
1367379
1368249
173286
173207
173248
1367963
1367861


In [31]:
# add lake column to species ocurrence table
gbif_snap_result <- merge(gbif_snap_result, lake_id[, c("id", "Hylak_id")], by = "id", all.x = TRUE)

# check which point occurrence IDs are located in lakes
indx <- which(!gbif_snap_result$Hylak_id == 0)
inside_lake <- gbif_snap_result[c(indx), ]

str(inside_lake)
head(inside_lake)

# lakes containing species points
unique(inside_lake$Hylak_id)

Classes ‘data.table’ and 'data.frame':	83 obs. of  10 variables:
 $ id                       : int  57 58 59 60 61 79 82 83 84 86 ...
 $ gbifID                   :integer64 4046214680 5133580071 1609379246 3774826676 5133772622 4976238061 1609379305 1609379247 ... 
 $ subc_id                  : int  561154454 561145731 561164732 561154454 561154454 561410234 561146248 561149143 561182341 561155879 ...
 $ strahler                 : int  4 4 6 4 4 6 4 4 5 4 ...
 $ decimalLongitude_snapped : num  21.5 21.5 21.5 21.5 21.5 ...
 $ decimalLongitude_original: num  21.5 21.5 21.5 21.5 21.5 ...
 $ decimalLatitude_snapped  : num  38.6 38.6 38.6 38.6 38.6 ...
 $ decimalLatitude_original : num  38.6 38.6 38.6 38.6 38.6 ...
 $ distance_metres          : num  47.9 49.4 235.2 390.9 366.7 ...
 $ Hylak_id                 : int  14679 14679 14679 14679 14679 173526 14679 14679 14679 14679 ...
 - attr(*, ".internal.selfref")=<externalptr> 
 - attr(*, "sorted")= chr "id"


id,gbifID,subc_id,strahler,decimalLongitude_snapped,decimalLongitude_original,decimalLatitude_snapped,decimalLatitude_original,distance_metres,Hylak_id
<int>,<int64>,<int>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
57,4046214680,561154454,4,21.47125,21.4718,38.58670,38.5867,47.92045,14679
58,5133580071,561145731,4,21.49175,21.4914,38.59075,38.5904,49.38994,14679
59,1609379246,561164732,6,21.46167,21.4600,38.56833,38.5700,235.21629,14679
60,3774826676,561154454,4,21.47125,21.4668,38.58875,38.5892,390.91264,14679
61,5133772622,561154454,4,21.47125,21.4671,38.58875,38.5893,366.68849,14679
79,4976238061,561410234,6,21.38523,21.3845,38.15893,38.1582,103.71465,173526


[1]   14679  173526  173778   14615   14654   14533   14507  172681 1371681
[10]  172557   14476   14511   14514   14478  172405   14686  173522  172980
[19] 1370799   14587  173348  173988 1368040 1367650    1312  172605 1369472
[28]  173286  173207 1367924  172570

In total 83 fish occurrences are found in 31 lakes within our study area.

![](./img/lakes_with_species.PNG)

## Identifying the intersection points between stream network and lakes

With the information of which lakes are inside our study area, we can now continue and identify the intersection points of our Hydrography90m
stream network and those lakes. To speed things up we will concentrate on the first five lakes containing species occurrence points (maybe just looking at one basin?).

**NOTE:** For the get_lake_intersection(); we need the open source MSPA analysis tool from GuidosToolbox Workbench (Vogt, P. & Riitters, K., 2017; Vogt, P. & Soille, P., 2009).
You can find more information about the tool [here](https://forest.jrc.ec.europa.eu/en/activities/lpa/gwb/).

In [32]:

# subset lake data to first five lakes
lake_ids <- inside_lake[1:5, ]

# show selected lake IDs
print(lake_ids)

Key: <id>
      id     gbifID   subc_id strahler decimalLongitude_snapped
   <int>      <i64>     <int>    <int>                    <num>
1:    57 4046214680 561154454        4                 21.47125
2:    58 5133580071 561145731        4                 21.49175
3:    59 1609379246 561164732        6                 21.46167
4:    60 3774826676 561154454        4                 21.47125
5:    61 5133772622 561154454        4                 21.47125
   decimalLongitude_original decimalLatitude_snapped decimalLatitude_original
                       <num>                   <num>                    <num>
1:                   21.4718                38.58670                  38.5867
2:                   21.4914                38.59075                  38.5904
3:                   21.4600                38.56833                  38.5700
4:                   21.4668                38.58875                  38.5892
5:                   21.4671                38.58875                  38.5

All our 5 points fall into the same lake which is lake Trichonida. We will now identify the intersection points between lake Trichonida and our stream network.
Do to this we need to define the path to our previously generated merged raster files and set the path to the MSPA analysis tool.

In [38]:
# lets load in our previously downloaded raster data
# needed for the get_lake_intersection() function

# merged stream network raster file
stream_raster <- paste0("./lakes", "/hydrography90m/merge_segment.tif")

# merged flow accumulation raster file
flow_raster <- paste0("./lakes", "/hydrography90m/merge_accumulation.tif")

# merged basin raster file
basin_raster <- paste0("./lakes", "/hydrography90m/merge_basin.tif")

# give full path virtual machine linux path
gwb <- "/home/participant/GWB2.0.0/GWB"

Now we can run the get_lake_intersection() function which returns us the following outputs (in brackets generic output names):

1. lake point intersection table (coord_lake_[ID].txt)
2. lake raster file (lake_[ID].tif)
3. lake extent (lakes_ref_extent_[ID].txt)

NOTE: all these files are needed for the lake catchment delineation with the get_lake_catchment() function.

Lets create a folder to save the outputs of get_lake_intersection().

In [36]:
# Create a new data folder in the working directory to store all the data
dir.create(paste0("./lakes", "/lake_intersection"))

In [41]:
# get all intersection points between lake Trichonida and the stream network
get_lake_intersection(data = lake_ids,
                      lakes = hydro_lakes,
                      lake_id = "Hylak_id",
                      lake_name = "HydroLAKES_polys_v10",
                      buffer = TRUE,
                      edge = gwb,
                      stream = stream_raster,
                      flow = flow_raster,
                      basins = basin_raster,
                      lake_dat = paste0("./lakes", "/lake_intersection"),
                      n_cores = 1,
                      quiet = FALSE)


/opt/conda/lib/R/library/hydrographr/sh/get_lake_intersection.sh: 56: Syntax error: "(" unexpected (expecting "then")


ERROR: Error in c("processx::run(system.file(\"sh\", \"get_lake_intersection.sh\", package = \"hydrographr\"), ", : [33m![39m System command 'get_lake_intersection.sh' failed


![](./img/lake_intersection_outlet.PNG)

Lets remove objects no longer needed after get_lake_intersection()

In [ ]:
# rm(stream_raster,   # used only as input to get_lake_intersection()
#   flow_raster,     # used only as input to get_lake_intersection()
#   basin_raster,    # used only as input to get_lake_intersection()
#   gwb,             # GWB tool path, only needed for get_lake_intersection()
#   vars_tif,        # variable names used for downloading and merging
#   lake_ids)        # subset of lake IDs passed to get_lake_intersection()

# gc()

Now lets inspect the lake intersection table of lake Trichonida; HydroLAKES ID 14679.

In [3]:

lake_intersection_14679 <- fread(paste0(read_dir, "lakes/lake_intersection/coord_lake_14679.txt"))

# first rows of the lake intersection table
head(lake_intersection_14679)

outlet_ID,lake_ID,lon,lat,subc_id,flow_accu,flow_accu_max,flow_accu_mean
<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
1,14679,21.44208,38.55208,561174656,403.052,403.144,134.370
2,14679,21.61375,38.57458,561160799,24.708,25.146,11.074
3,14679,21.47875,38.54625,561177203,29.399,29.716,10.449
4,14679,21.54292,38.60042,561132990,18.447,18.447,6.293
5,14679,21.49042,38.59292,561145731,17.533,17.533,5.847
6,14679,21.51958,38.54958,561175151,9.963,13.689,5.448


Lake Trichonida has 116 stream segments intersecting with the Hydrography90m stream network. The intersection point represents the lake outlet, meaning that
all upstream contributing area of the lake drains into this point.

## Delineating the lake catchment 

We will now delineate the **upstream catchment area** of the lake outlet point, from now on referred to as the lake catchment area.
We first load the necessary data inputs for the get_lake_catchment() function.
NOTE: that the **lake raster file** `lake_[ID].tif` and **lake extent file** `lakes_ref_extent_[ID].txt` previously generated by the get_lake_intersection()
function need to be in the same folder as the lake intersection table and in which the lake_catchment is being stored.

In [4]:
direction_raster <- paste0(read_dir, "lakes/hydrography90m/merge_direction.tif")

Lets check if all the required files are in the same folder

In [ ]:
required_files <- c(
  paste0("./lakes", "/lake_intersection/coord_lake_14679.txt"),
  paste0("./lakes", "/lake_intersection/lake_14679.tif"),
  paste0("./lakes", "/lake_intersection/lakes_ref_extent_14679.txt")
)

missing_files <- required_files[!file.exists(required_files)]

if (length(missing_files) > 0) {
  stop("Missing required files for get_lake_catchment():\n",
       paste(" -", missing_files, collapse = "\n"))
} else {
  cat("All required files found in:", paste0("./lakes", "/lake_intersection"), "\n")
}

The parameter "n" represents the number intersection points for which lake catchments are being delineated, starting from the first row. 
**NOTE:** that in our case the lake outlet is the first row of the lake intersection table (by default sorted by mean flow accumulation);
meaning `n = 1` will only delineate the lake catchment for this point.
You can filter or sort the lake intersection table differently if you want the catchment area of an other intersection point or simply
`n = "all"` for all intersection points catchment areas.

In [10]:
file.copy(from = c(paste0(read_dir, "lakes/lake_intersection/lakes_ref_extent_14679.txt"),
                   paste0(read_dir, "lakes/lake_intersection/lake_14679.tif")),
          to   = "./lakes/lake_intersection")

[1] TRUE TRUE

In [7]:
# delineate the lake catchment from the lake outlet of lake Trichonida
get_lake_catchment(data = lake_intersection_14679,
                   direction = direction_raster,
                   flow = "flow_accu_mean",
                   lake_basin = paste0("./lakes", "/lake_intersection/"),
                   lake_id = "lake_ID",
                   n = 1,
                   n_cores = 1,
                   quiet = FALSE)

---------------------------------
CREATING UPSTREAM BASINS IN INTERSECTIONS FOR LAKE 14679 AND OUTLET
---------------------------------
/tmp/Rtmph3OrXH/ids_c2RVo5uC.txt
/tmp/Rtmph3OrXH/lake_data.txt
Creating output file that is 1011P x 1663L.
Processing ./lakes/lake_intersection//lake_14679.tif [1/1] : 0Using internal nodata values (e.g. 0) for image ./lakes/lake_intersection//lake_14679.tif.
Copying nodata values from source ./lakes/lake_intersection//lake_14679.tif to destination /tmp/Rtmph3OrXH/lake_14679_NA.tif.
...10...20...30...40...50...60...70...80...90...100 - done.
Creating output file that is 1011P x 1663L.
Processing /home/participant/data_readonly/lakes/hydrography90m/merge_direction.tif [1/1] : 0Using internal nodata values (e.g. 0) for image /home/participant/data_readonly/lakes/hydrography90m/merge_direction.tif.
Copying nodata values from source /home/participant/data_readonly/lakes/hydrography90m/merge_direction.tif to destination /tmp/Rtmph3OrXH/dir_14679_NA.tif.
...

The output of the get_lake_catchment() function are (in brackets generic output names):
1. the lake catchment raster file (basin_lake_[ID]_coord_[intersection_point_number].tif)
2. the intersection points (outlets_[ID].gpkg) stored in a geopackage

![](./img/lake_catchment.PNG)

Lets calculate the area of the lake catchment.

In [ ]:
lake_catch <- terra::rast(paste0("./lakes", "/lake_intersection/basin_lake_14679_coord_1.tif"))
# Calculate the lake area
# reproject to LAEA Europe (suitable for Greece)
lake_catch_laea <- terra::project(lake_catch, "EPSG:3035")

lake_catch_area <- terra::expanse(lake_catch_laea, unit = "km", zones = lake_catch_laea)

print(lake_catch_area)

## Calculating the broadleaf deciduous land cover using the Env90m dataset for the years 1992 - 2020

For this exercise we will use the previously downloaded land cover variable c60 from the Environment90m dataset (García Márquez, J. R., et al. 2026).
First we identify all sub-catchments of the lake catchment using the `extract_ids()` function. 
NOTE: here we use the previously downloaded and merged sub-catchment raster file.

In [ ]:
# load in sub-catchment raster file
subc_raster <- terra::rast(paste0("./lakes", "/hydrography90m/merge_sub_catchment.tif"))

# crop the sub-catchment raster file to the extent of the lake catchment
lake_catch_crop <- terra::crop(subc_raster, lake_catch)

# extract the sub-catchment information for the lake catchment
lake_catch_crop <- terra::mask(lake_catch_crop, lake_catch)

# we need to write the cropped catchment raster file to disk first
writeRaster(lake_catch_crop, paste0("./lakes", "/lake_intersection/subc_id_lake_catchment.tif"))

# second we can extract the sub-catchment IDs for the lake catchment
lake_catch_ids <- extract_ids(subc_layer = paste0("./lakes", "/lake_intersection/subc_id_lake_catchment.tif"))

# write the sub-catchment IDs to disk
fwrite(lake_catch_ids, paste0("./lakes", "/subc_IDs.txt"))

Now we subset the Environment90m land cover data by the sub_catchment ID and join the yearly data table to create a time series of land cover change.

In [ ]:
years <- 1992:2020
tile_id <- "h20v04"

# Create name vectors
c60_names <- paste0("c60_", years)

# name of the land cover variable
var <- c60_names

tb <- get_predict_table(variable = var,
                        statistics = c("mean"),
                        tile_id = tile_id,
                        input_var_path = paste0(read_dir, "/esa_cci_landcover_v2_1_1/"),
                        subcatch_id = file.path("./lakes", "/subc_IDs.txt"),
                        out_file_path = paste0("./lakes", "/predictTB.csv"),
                        read = TRUE,
                        overwrite = TRUE,
                        n_cores = 1)

# show result of get_predict_table()
head(tb)

# write the land cover of the lake catchment area to disk
fwrite(tb, paste0("./lakes", "/land_cover_subc.txt"))

The land cover values show the percentage covered by the different land use categories. To show a change in land cover we will convert
the percentage in the area they cover of any given sub-catchment. For this we will calculate the area of our sub-catchments.

In [ ]:

# load in the sub-catchment raster file of the lake catchment area
subc_raster <- rast(paste0("./lakes", "/lake_intersection/subc_id_lake_catchment.tif"))
subc_areas <- terra::expanse(subc_raster, unit = "km", zones = subc_raster)
names(subc_areas) <- c("layer", "subc_id", "area_km2")

# load in the land cover proportion per sub-catchment
tb <- fread(paste0("./lakes", "/land_cover_subc.txt"))
names(tb)[names(tb) == "subcID"] <- "subc_id"

# merge areas with land cover table
tb <- merge(tb, subc_areas[, c("subc_id", "area_km2")], by = "subc_id")

# multiply each proportion by its sub-catchment area to get km²
lc_cols <- grep("^c60_", names(tb))
tb_area <- tb[, ..lc_cols] * tb$area_km2

# sum across all sub-catchments per year
col_sums <- colSums(tb_area)

# convert to data frame for plotting
lake_land_cover <- enframe(col_sums, name = "id", value = "area_km2")
lake_land_cover <- lake_land_cover %>%
  separate(id, into = c("variable", "year"), sep = "_y") %>%
  mutate(year = as.numeric(year))


Lets portray the land cover time series within a simple histogram, showcasing the change over time in the land cover variable XY in the lake catchment. 

In [ ]:
# Plot
ggplot(lake_land_cover, aes(x = year, y = area_km2, color = variable)) +
  geom_line(linewidth = 1.2) +
  scale_color_manual(
    values = c("c60" = "darkgreen"),
    labels = c("c60" = "Tree cover")
  ) +
  scale_x_continuous(
    breaks = seq(min(lake_land_cover$year), max(lake_land_cover$year), by = 2)
  ) +
  scale_y_continuous(
    labels = scales::comma
  ) +
  labs(x = "Year", y = "Tree cover area (km²)", color = "") +
  theme_minimal(base_size = 14) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "top"
  )

## Calculating the upstream and downstream distance 

Knowing the intersection points unique stream segment IDs of our network enables us to perform basic network analyses like for instance
calculating the upstream and downstream distance from a lakes outlet point.

In [ ]:

# tp get the upstream length from all the upstream stream segments we can simply subset out stream network
# and sum up the pre calculated length (in meter) column
total_length <- sum(
  stream_network$length[stream_network$subc_id %in% lake_catch_ids$subcatchment_id]
) / 1000

cat("Total stream length:", total_length, "km\n")


In [ ]:
# get first the sub-catchment ID of the lake outlet
lake_outlet_id <- as.numeric(lake_intersection_14679[1, "subc_id"])
cat("Lake outlet ID:", lake_outlet_id, "\n")

# then filter to segments within the catchment that downstream of the lake outlet
current_id <- lake_outlet_id
max_strahler <- 0
trace <- list()

repeat {
  row <- stream_network[stream_network$subc_id == current_id, ]

  if (nrow(row) == 0) break

  # Stop if strahler order decreases
  if (row$strahler_order < max_strahler) break

  max_strahler <- row$strahler_order
  trace[[length(trace) + 1]] <- row

  # Stop if target is negative (reached outlet)
  if (row$target < 0) break

  # Move to next segment
  current_id <- as.numeric(row$target)
}

network_outlet <- trace[[length(trace)]]
stream_network_outlet_id <- as.numeric(network_outlet$subc_id)

cat("Network outlet:", stream_network_outlet_id, "\n")
cat("Strahler order:", network_outlet$strahler_order, "\n")
cat("Steps downstream:", length(trace), "\n")

# make sure both are of numeric data type
lake_to_stream_outlet <- as.numeric(c(lake_outlet_id, stream_network_outlet_id))


# Calculate the distance between the lake outlet and the stream network outlet
downstream_distance <- get_distance_graph(stream_network_graph,
                                          subc_id = lake_to_stream_outlet,
                                          variable = "length",
                                          distance_m = TRUE)

head(downstream_distance)

It takes around 72 km from the lake outlet to the stream network outlet to the ocean. And it in total all upstream stream segments sum up to a stream length of 408 km.

For all downstream and upstream segments we calculate the total stream length in meters using the get_distance() function.

## Conclusions 

With our hydrographr (lake) functions & datasets we are able to:
1. Identify lakes within our study area
2. Identify the intersection points between our stream network and lakes
3. Delineate the lake catchment area from each intersection point
4. Perform land cover time series analyses
5. Perform basic network analyses